Session 2 RQ2 recovered inference v4. Offline exact canonical model files and dependencies; frozen shared INPUT snapshot; one idle H100 (prefer physical3), float32. Original shared prompts/parser with only the noniterable TypeError guard. Real smoke3 then full100 B and C. I/O correction: allow only the exact authorized scratch snapshot symlink target in addition to OWN.

In [ ]:
from pathlib import Path
import os, sys, json, time, hashlib, importlib.util, subprocess, platform, random, traceback, importlib.metadata
from datetime import datetime, timezone
ROOT=Path.cwd()
OWN=ROOT/"llm4rec_s2_rq2"
INPUT=OWN/"inputs"
ART=OWN/"recovered_v1"
ART.mkdir(exist_ok=True)
input_spec_path=OWN/"inference_inputs_v3.json"
SPEC=json.loads(input_spec_path.read_text())
assert SPEC["revision"]=="70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"
AUTHORIZED_SCRATCH_SNAPSHOT=Path("/tmp/REC-23097-night-20261116-v1/llm4rec_s2_rq2/model_snapshot_70d_v1")
def own_path(relative):
    p=(OWN/relative).resolve()
    assert p.is_relative_to(OWN.resolve()) or p.is_relative_to(AUTHORIZED_SCRATCH_SNAPSHOT), "Path leaves exact authorized own roots"
    return p
SNAPSHOT=own_path(SPEC["snapshot_relative_path"])
api_path=own_path(SPEC["canonical_api_relative_path"])
tree_path=own_path(SPEC["canonical_tree_relative_path"])
api=json.loads(api_path.read_text());tree=json.loads(tree_path.read_text())
assert api["id"]=="Qwen/Qwen3-1.7B" and api["sha"]==SPEC["revision"] and api["private"] is False
# Verify canonical LFS SHA256, Git blob SHA1 for every assembled canonical file, and required model files.
t0=time.perf_counter()
verified=[]
required={"config.json","generation_config.json","merges.txt","model-00001-of-00002.safetensors","model-00002-of-00002.safetensors","model.safetensors.index.json","tokenizer.json","tokenizer_config.json","vocab.json"}
for entry in tree:
    if entry["type"]!="file":continue
    p=SNAPSHOT/entry["path"]
    if not p.is_file():
        assert entry["path"] not in required and "lfs" not in entry,"Missing required pinned model file: "+entry["path"]
        continue
    assert p.stat().st_size==entry["size"]
    sha256=hashlib.sha256();git=hashlib.sha1()
    git.update(("blob "+str(p.stat().st_size)+"\0").encode())
    with p.open("rb") as f:
        for block in iter(lambda:f.read(8*1024**2),b""):
            sha256.update(block);git.update(block)
    actual=sha256.hexdigest() if "lfs" in entry else git.hexdigest()
    expected=entry["lfs"]["oid"] if "lfs" in entry else entry["oid"]
    assert actual==expected,entry["path"]+" canonical hash mismatch"
    verified.append({"path":entry["path"],"bytes":p.stat().st_size,"canonical_method":"sha256" if "lfs" in entry else "git_blob_sha1","expected":expected,"actual":actual,"sha256":sha256.hexdigest()})
assert required<={r["path"] for r in verified}
verification={"canonical_revision":api["sha"],"canonical_api_sha256":hashlib.sha256(api_path.read_bytes()).hexdigest(),"canonical_tree_sha256":hashlib.sha256(tree_path.read_bytes()).hexdigest(),"files":verified,"hash_verification_seconds":time.perf_counter()-t0}
verification_digest=hashlib.sha256(json.dumps({k:v for k,v in verification.items() if k!="hash_verification_seconds"},sort_keys=True).encode()).hexdigest()
(ART/"pinned-snapshot-verification.json").write_text(json.dumps(verification,indent=2)+"\n")
os.environ["HF_HUB_OFFLINE"]="1"
os.environ["TRANSFORMERS_OFFLINE"]="1"
os.environ["HF_HUB_DISABLE_TELEMETRY"]="1"
os.environ["USE_TF"]="0"
os.environ["USE_FLAX"]="0"
sys.path.insert(0,str(OWN/"offline_packages_v1"))
import torch, transformers, numpy as np
import transformers.utils.import_utils as import_utils
for flag in ["_librosa_available","_torchvision_available","_torchaudio_available"]:setattr(import_utils,flag,False)
from transformers import AutoTokenizer,AutoModelForCausalLM,AutoConfig
assert torch.__version__=="2.9.1+cu128" and transformers.__version__=="4.57.6"
assert hashlib.sha256((INPUT/"llm_rerank_original.py").read_bytes()).hexdigest()=="d7a34c1f9f715d7437ca5b2cc7989dbff12348602495b8a427cf8e56fa7007ba"
spec=importlib.util.spec_from_file_location("shared",INPUT/"llm_rerank_original.py")
shared=importlib.util.module_from_spec(spec);spec.loader.exec_module(shared)
contexts,protocol=shared.load_run(OWN/"shared_contexts")
context_sha256=hashlib.sha256((OWN/"shared_contexts/contexts.jsonl").read_bytes()).hexdigest()
assert context_sha256=="a505745968316fb96f03a453726878d84501965a12482e81b54b8bf468a1093e"
assert len(contexts)==100 and [c["user_id"] for c in contexts]==protocol["users"]
assert protocol["protocol_hash"].startswith("50efd4aeb415")
parity=json.loads((OWN/"shared-context-parity.json").read_text())
assert parity["candidate_sets_equal"] and parity["sha256"]["contexts.jsonl"]==context_sha256
for ctx in contexts:
    shared.assert_no_future(ctx)
    assert len(ctx["candidate_assets"])==len({a["id"] for a in ctx["candidate_assets"]})==50
    assert {a["id"] for a in ctx["candidate_assets"]}<=set(ctx["eligible_asset_ids"])

def parse_guarded(answer, n):
    try:
        return shared.parse_ranking(answer, n), None
    except TypeError as error:
        # The unchanged shared parser iterates ranking:null/scalar and raises TypeError.
        # These are invalid outputs; [] lets the unchanged starter apply CBF fallback.
        return [], {"type": "TypeError", "message": str(error), "guard": "noniterable ranking"}

parser_checks = []
for text in ['{"ranking": null}', '{"ranking": 1}', '{"ranking": 1.2}', '{"ranking": true}']:
    raised = False
    try: shared.parse_ranking(text, 50)
    except TypeError: raised = True
    kept, guard = parse_guarded(text, 50)
    assert raised and kept == [] and guard
    parser_checks.append({"response": text, "original_raises_TypeError": raised, "guarded": kept})
for text in ['{"ranking": [1,2,3,4,5,6,7,8,9,10]}', '{"ranking": [true,0,51,2,2,1,"3"]}', 'bad', '[]', '{"ranking": "abc"}', '{"ranking": {"a":1}}']:
    assert parse_guarded(text, 50)[0] == shared.parse_ranking(text, 50)
    parser_checks.append({"response": text, "original_behavior_preserved": True})
(ART / "parser-checks.json").write_text(json.dumps(parser_checks, indent=2) + "\n")

MODEL="Qwen/Qwen3-1.7B"
REVISION="70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"
CONFIG={"model":MODEL,"revision":REVISION,"dtype":"float32","seed":0,"do_sample":False,"enable_thinking":False,"summary_max_new_tokens":120,"rank_max_new_tokens":60,"device":"cuda:0","tf32":False,"torch":torch.__version__,"transformers":transformers.__version__,"huggingface-hub":importlib.metadata.version("huggingface-hub"),"requirements_sha256":hashlib.sha256((OWN/"requirements-inference-actual.txt").read_bytes()).hexdigest(),"frozen_contexts_sha256":context_sha256,"frozen_protocol_file_sha256":hashlib.sha256((OWN/"shared_contexts/protocol.json").read_bytes()).hexdigest(),"snapshot_verification_digest":verification_digest,"inference_input_spec_sha256":hashlib.sha256(input_spec_path.read_bytes()).hexdigest()}
CACHE=ART/"generation-cache"
CACHE.mkdir(exist_ok=True)
def save_new(path,obj):
    with path.open("x",encoding="utf-8") as f:
        json.dump(obj,f,ensure_ascii=False);f.write("\n");f.flush();os.fsync(f.fileno())
def now():return datetime.now(timezone.utc).isoformat()
class ContextOverflow(RuntimeError):pass

def select_idle_h100():
    while True:
        p=subprocess.run(["nvidia-smi","--query-gpu=index,name,uuid,driver_version,memory.total,memory.used,utilization.gpu","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True)
        rows=[[s.strip() for s in line.split(",")] for line in p.stdout.strip().splitlines()]
        observation={"observed_at_utc":now(),"all_gpu_observation":p.stdout,"policy":"one H100, prefer physical3 then2 then1 then0; memory<100MiB and utilization0"}
        idle={int(r[0]):r for r in rows if "H100" in r[1] and int(r[-2])<100 and int(r[-1])==0}
        for index in [3,2,1,0]:
            if index in idle:return idle[index],observation
        (ART/"allocation-wait.json").write_text(json.dumps(observation,indent=2)+"\n")
        print("All H100s occupied; waiting without allocating or stopping any job.",flush=True)
        time.sleep(30)

class GPUBackend:
    def __init__(self):
        t0=time.perf_counter()
        self.tok=AutoTokenizer.from_pretrained(str(SNAPSHOT),local_files_only=True)
        cfg=AutoConfig.from_pretrained(str(SNAPSHOT),local_files_only=True)
        assert cfg.model_type=="qwen3" and cfg.max_position_embeddings==40960
        self.context_limit=int(cfg.max_position_embeddings)
        token_checks=[]
        for ctx in contexts:
            for kind,prompt,cap in [("B_rank",shared.rank_prompt(ctx,shared.history_block(ctx)),60),("C_summary",shared.summary_prompt(ctx,"C",False),120)]:
                system,user=prompt
                rendered=self.tok.apply_chat_template([{"role":"system","content":system},{"role":"user","content":user}],tokenize=False,add_generation_prompt=True,enable_thinking=False)
                assert rendered.endswith("<think>\n\n</think>\n\n")
                tokens=self.tok(rendered,return_tensors="pt")["input_ids"].shape[1]
                assert tokens+cap<=self.context_limit,"Preflight context overflow"
                token_checks.append({"user_id":ctx["user_id"],"purpose":kind,"input_tokens":tokens,"cap":cap})
        save_new(ART/"tokenization-validation.json",{"local_files_only":True,"thinking_off_template_verified":True,"model_type":cfg.model_type,"tokens":token_checks,"validation_seconds":time.perf_counter()-t0})
        # No CUDA allocation has happened. Choose the one physical GPU immediately before loading.
        assert not torch.cuda.is_initialized()
        fields,observation=select_idle_h100()
        os.environ["CUDA_VISIBLE_DEVICES"]=fields[2]
        os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
        random.seed(0);np.random.seed(0);torch.manual_seed(0)
        torch.set_float32_matmul_precision("highest")
        torch.backends.cuda.matmul.allow_tf32=False
        torch.backends.cudnn.allow_tf32=False
        load_start=time.perf_counter()
        self.model=AutoModelForCausalLM.from_pretrained(str(SNAPSHOT),local_files_only=True,torch_dtype=torch.float32)
        # Recheck all GPUs directly before moving model/tensors to the selected device.
        p=subprocess.run(["nvidia-smi","--query-gpu=index,name,uuid,driver_version,memory.total,memory.used,utilization.gpu","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True)
        fresh_rows=[[s.strip() for s in line.split(",")] for line in p.stdout.strip().splitlines()]
        selected=next(r for r in fresh_rows if r[2]==fields[2])
        observation.update(immediately_before_cuda_allocation_at_utc=now(),immediately_before_cuda_allocation_all_gpus=p.stdout,selected_physical_index=int(fields[0]),selected_uuid=fields[2],CUDA_VISIBLE_DEVICES=os.environ["CUDA_VISIBLE_DEVICES"])
        save_new(ART/"gpu-before-allocation.json",observation)
        assert int(selected[-2])<100 and int(selected[-1])==0,"Selected GPU became occupied during CPU load; no allocation performed, inspect before recovery"
        assert torch.cuda.is_available() and torch.cuda.device_count()==1
        cpu_load_seconds=time.perf_counter()-load_start
        move_start=time.perf_counter()
        self.model=self.model.to("cuda:0");self.model.eval();torch.cuda.synchronize()
        self.load_seconds=time.perf_counter()-load_start
        assert str(self.model.dtype)=="torch.float32" and {str(p.device) for p in self.model.parameters()}=={"cuda:0"}
        CONFIG.update(attention_implementation=self.model.config._attn_implementation,physical_gpu_uuid=fields[2],physical_gpu_index=int(fields[0]))
        self.meta={**CONFIG,"requested_revision":REVISION,"resolved_commit_via_canonical_file_hashes":api["sha"],"model_config_commit_hash":getattr(self.model.config,"_commit_hash",None),"snapshot_path":str(SNAPSHOT),"local_files_only":True,"context_limit":self.context_limit,"tokenizer_model_max_length":self.tok.model_max_length,"model_generation_config":self.model.generation_config.to_dict(),"cold_model_load_seconds":self.load_seconds,"CPU_model_load_seconds":cpu_load_seconds,"model_move_to_GPU_seconds":time.perf_counter()-move_start,"tokenizer_validation_seconds":load_start-t0,"cuda_runtime":torch.version.cuda,"cudnn_version":torch.backends.cudnn.version(),"tf32_matmul":torch.backends.cuda.matmul.allow_tf32,"tf32_cudnn":torch.backends.cudnn.allow_tf32,"float32_matmul_precision":torch.get_float32_matmul_precision(),"hardware":{"gpu_name":torch.cuda.get_device_name(0),"device_count_visible":torch.cuda.device_count(),"physical_gpu_index":int(fields[0]),"gpu_uuid":fields[2],"driver_version":fields[3],"platform":platform.platform(),"hostname":platform.node(),"python":sys.version,"kernel_executable":sys.executable,"offline_packages":str(OWN/"offline_packages_v1"),"cpu_threads":torch.get_num_threads()},"loaded_dependency_paths":{"torch":torch.__file__,"transformers":transformers.__file__,"numpy":np.__file__},"torch_deviation":"Authorized existing torch2.9.1+cu128 instead of teammates torch2.11 CPU; B and C share this exact adapter/model/dependencies; no teammate model scores used"}
        save_new(ART/"model-hardware-metadata.json",self.meta)
        print(json.dumps(self.meta,indent=2),flush=True)

    def chat(self, system, user, max_new_tokens):
        messages = [{"role":"system", "content":system}, {"role":"user", "content":user}]
        rendered = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        enc = self.tok(rendered, return_tensors="pt")
        input_ids = enc["input_ids"][0].tolist()
        request = {"config": CONFIG, "messages": messages, "max_new_tokens": max_new_tokens, "rendered_chat": rendered}
        key = hashlib.sha256(json.dumps(request, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
        path = CACHE / (key + ".json")
        if len(input_ids) + max_new_tokens > self.context_limit:
            failure = {"key": key, "request": request, "input_tokens": len(input_ids), "context_limit": self.context_limit, "failure": "context overflow"}
            save_new(CACHE / (key + ".overflow.json"), failure)
            raise ContextOverflow(json.dumps(failure))
        if path.exists():
            record = json.loads(path.read_text())
            assert record["request"] == request and record["input_ids"] == input_ids
            return {**record, "cache_reused": True}
        t0 = time.perf_counter()
        enc = {k:v.to("cuda:0") for k,v in enc.items()}
        torch.cuda.synchronize()
        generation_start = time.perf_counter()
        try:
            with torch.no_grad():
                out = self.model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=self.tok.eos_token_id)
            torch.cuda.synchronize()
        except Exception as error:
            save_new(CACHE / (key + ".failure-" + str(time.time_ns()) + ".json"), {"request": request, "input_ids": input_ids, "error": repr(error), "traceback": traceback.format_exc(), "at_utc": now()})
            raise
        generation_seconds = time.perf_counter()-generation_start
        output_ids = out[0][enc["input_ids"].shape[1]:].tolist()
        text = self.tok.decode(output_ids, skip_special_tokens=True).strip()
        record = {"key": key, "request": request, "input_ids": input_ids, "output_ids": output_ids,
            "input_tokens": len(input_ids), "output_tokens": len(output_ids), "context_limit": self.context_limit,
            "hit_new_token_cap": len(output_ids) == max_new_tokens, "response": text,
            "generation_seconds": generation_seconds, "transfer_plus_generation_seconds": time.perf_counter()-t0,
            "at_utc": now(), "cache_reused": False}
        save_new(path, record)
        return record

backend=GPUBackend()


In [ ]:
def prompt_checks(ctx, system, user, kind, summary=None):
    # Projection removes names/profile/metadata fields; equality proves they cannot affect actual prompts.
    projected = {"cutoff":ctx["cutoff"], "history":[{k:a[k] for k in ("id","attributes","buy_count")} for a in ctx["history"]],
                 "candidate_assets":[{k:a[k] for k in ("id","attributes")} for a in ctx["candidate_assets"]]}
    if kind == "summary": expected = shared.summary_prompt(projected, "C", False)
    else: expected = shared.rank_prompt(projected, shared.history_block(projected) if summary is None else summary)
    assert (system,user) == expected
    for field in ["riskLevel", "investmentCapacity", "customerType", "lastQuestionnaireDate", "assetName", '"truth"', '"label"', '"future"']:
        assert field not in system + user, "Prohibited input field: " + field
    return {"shared_template_exact":True, "projection_without_name_profile_future_fields_exact":True}

def run_arm(arm, selected, label):
    assert arm in ["B","C"]
    out_dir = ART / label / arm
    out_dir.mkdir(parents=True, exist_ok=False)
    t0 = time.perf_counter()
    logs, responses = [], []
    with (out_dir / "raw_log.jsonl").open("x") as raw, (out_dir / "responses.jsonl").open("x") as resp:
        for ctx in selected:
            shared.assert_no_future(ctx)
            log = {"user_id": ctx["user_id"], "arm": arm, "protocol_hash": protocol["protocol_hash"],
                   "candidate_ids": [a["id"] for a in ctx["candidate_assets"]], "summary": None,
                   "summary_generation": None, "rank_generation": None, "failures": []}
            numbers, guard = [], None
            try:
                if arm == "B": info = shared.history_block(ctx)
                else:
                    s_sys,s_user = shared.summary_prompt(ctx, "C", False)
                    log["summary_system"],log["summary_prompt"] = s_sys,s_user
                    log["summary_prompt_checks"] = prompt_checks(ctx,s_sys,s_user,"summary")
                    sg = backend.chat(s_sys,s_user,120)
                    log["summary_generation"] = sg
                    log["summary"] = info = sg["response"]
                r_sys,r_user = shared.rank_prompt(ctx,info)
                log["rank_system"],log["rank_prompt"] = r_sys,r_user
                log["rank_prompt_checks"] = prompt_checks(ctx,r_sys,r_user,"rank",log["summary"])
                rg = backend.chat(r_sys,r_user,60)
                log["rank_generation"] = rg
                log["rank_output"] = rg["response"]
                numbers,guard = parse_guarded(rg["response"],len(ctx["candidate_assets"]))
            except ContextOverflow:
                raise
            except Exception as error:
                log["failures"].append({"type": type(error).__name__, "error": str(error), "traceback": traceback.format_exc()})
            if guard: log["failures"].append(guard)
            ids = [ctx["candidate_assets"][n-1]["id"] for n in numbers]
            assert len(ids) == len(set(ids)) and set(ids) <= set(log["candidate_ids"])
            log.update(parsed_numbers=numbers,ranked_ids=ids,valid=len(ids)==10)
            response = {"user_id":ctx["user_id"], "stage":"ranker", "protocol_hash":protocol["protocol_hash"], "ranked_ids":ids}
            raw.write(json.dumps(log,ensure_ascii=False)+"\n"); raw.flush(); os.fsync(raw.fileno())
            resp.write(json.dumps(response)+"\n"); resp.flush(); os.fsync(resp.fileno())
            logs.append(log); responses.append(response)
            if len(logs) % 10 == 0 or label == "smoke":
                progress = {"at_utc":now(), "label":label,"arm":arm,"done":len(logs),"total":len(selected),"invalid":sum(not x["valid"] for x in logs), "wall_seconds":time.perf_counter()-t0}
                (ART / "inference-progress.json").write_text(json.dumps(progress,indent=2)+"\n")
                print(json.dumps(progress),flush=True)
    calls = [x[k] for x in logs for k in ["summary_generation","rank_generation"] if x[k] is not None]
    meta = {"arm":arm,"label":label,"users":len(selected),"invalid_outputs":sum(not x["valid"] for x in logs),
        "protocol_hash":protocol["protocol_hash"], "wall_seconds_whole_arm_excluding_cold_load":time.perf_counter()-t0,
        "logical_generation_calls":len(calls), "cache_reused_calls":sum(x["cache_reused"] for x in calls),
        "fresh_generation_calls":sum(not x["cache_reused"] for x in calls),
        "fresh_gpu_generation_seconds":sum(x["generation_seconds"] for x in calls if not x["cache_reused"]),
        "logical_generation_seconds_including_original_cached_calls":sum(x["generation_seconds"] for x in calls),
        "input_tokens":sum(x["input_tokens"] for x in calls), "output_tokens":sum(x["output_tokens"] for x in calls),
        "max_input_tokens":max(x["input_tokens"] for x in calls),"hit_new_token_cap_calls":sum(x["hit_new_token_cap"] for x in calls),
        "parser_guard":"Catch only TypeError from unchanged parse_ranking; mark invalid [] for starter fallback",
        "generation":CONFIG,"source_sha256":os.environ.get("CART_LAUNCH_SOURCE_SHA256")}
    save_new(out_dir / "run_meta.json", meta)
    return logs

# Real smoke: no labels or metrics consulted, and no prompt/config changes based on responses.
smoke_B = run_arm("B",contexts[:3],"smoke")
smoke_C = run_arm("C",contexts[:3],"smoke")
assert len(smoke_B) == len(smoke_C) == 3
assert all(x["rank_generation"] is not None for x in smoke_B+smoke_C), "Real smoke generation failed"
assert all(x["summary_generation"] is not None for x in smoke_C)
save_new(ART / "smoke-check.json", {"real_model":True,"users":3,"B_invalid":sum(not x["valid"] for x in smoke_B),"C_invalid":sum(not x["valid"] for x in smoke_C),"prompts_changed_after_smoke":False,"truth_consulted":False})
print((ART / "smoke-check.json").read_text())


In [ ]:
full_B = run_arm("B",contexts,"full100")
full_C = run_arm("C",contexts,"full100")
assert [x["user_id"] for x in full_B] == [x["user_id"] for x in full_C] == protocol["users"]
assert [x["candidate_ids"] for x in full_B] == [x["candidate_ids"] for x in full_C]
save_new(ART / "inference-completed.json", {"at_utc":now(),"B_users":len(full_B),"C_users":len(full_C),"protocol_hash":protocol["protocol_hash"],"source_sha256":os.environ.get("CART_LAUNCH_SOURCE_SHA256")})
print((ART / "inference-completed.json").read_text())
